# 3 — The Digital Thread as a Role on the Portal Network

## The Problem With "Digital Thread as Description"

Most digital thread implementations model the thread as a static
description: "requirement A traces to design element B traces to
test result C."  This is a diagram, not an operational system.

The DEIX Ontology is more precise.  A `deix:Thread_Description_ICE`
is composed of `deix:Traceability_Measurement_ICE`s — each measuring
**consistency** between a Subject and a Referent.  And the
`deix:Digital_Thread_Role` is realized in `deix:Act_of_Measuring_Consistency`.

In holonic terms: the Digital Thread is NOT a named graph containing
trace links.  The Digital Thread IS A **ROLE** applied to the collection
of portals, their PROV-O traversal activities, and the consistency
validations that occurred at each boundary crossing.

The thread is the **emergent structure** of governed traversals.
It does not need to be separately maintained — it IS the provenance chain.

## The Pattern

```
The Digital Thread Role is realized in:
  - Portal traversal activities (prov:Activity)
  - Membrane validations at each boundary (SHACL results)
  - Provenance chains linking graph IRIs across holons

The Thread Description ICE is composed of:
  - Traceability measurements (portal validation outcomes)
  - Cross-holon derivation links (prov:wasDerivedFrom between graph IRIs)
```

In [1]:
from holonic import (
    Holon, TransformPortal, Holarchy,
    validate_membrane, MembraneHealth,
    ProvenanceTracker,
)
from rdflib import Graph, URIRef, RDF, Namespace
from io import StringIO

DEIX = Namespace("https://semantic.incose.org/DEIX_Ontology#")
CGA  = Namespace("urn:cga:")
PROV = Namespace("http://www.w3.org/ns/prov#")

PREFIXES = """
@prefix deix:  <https://semantic.incose.org/DEIX_Ontology#> .
@prefix cga:   <urn:cga:> .
@prefix prov:  <http://www.w3.org/ns/prov#> .
@prefix rdf:   <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs:  <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd:   <http://www.w3.org/2001/XMLSchema#> .
@prefix sh:    <http://www.w3.org/ns/shacl#> .
@prefix ex:    <urn:example:> .
"""

## Build a Three-Holon Pipeline and Record Everything

We build the same ToolX → DEIX → ToolY pipeline from notebook 02,
but this time we focus on the provenance chain that constitutes
the Digital Thread.

In [2]:
# --- Requirements tool ---
reqs = Holon(
    iri="urn:holon:tool:doors",
    label="DOORS (Requirements)",
    depth=1,
    interior_ttl="""
        @prefix req: <urn:req:> .
        <urn:req:REQ-MASS-001> a req:Requirement ;
            rdfs:label "Mass Limit" ;
            req:reqId "REQ-MASS-001" ;
            req:text "Subsystem mass shall not exceed 150 kg." ;
            req:priority "SHALL" ;
            req:threshold 150.0 ;
            req:unit "kg" .
    """,
    context_ttl="<urn:holon:tool:doors> cga:memberOf <urn:holon:domain:engineering> .",
)

# --- Design tool ---
design = Holon(
    iri="urn:holon:tool:sysml",
    label="SysML v2 (Design)",
    depth=1,
    interior_ttl="""
        @prefix sysml: <urn:sysml:> .
        <urn:sysml:block:tms> a sysml:Block ;
            sysml:name "ThermalMgmtSubsystem" .
        <urn:sysml:param:mass> a sysml:ValueProperty ;
            sysml:name "mass" ;
            sysml:value 142.3 ;
            sysml:unit "kg" ;
            sysml:owner <urn:sysml:block:tms> .
    """,
    context_ttl="<urn:holon:tool:sysml> cga:memberOf <urn:holon:domain:engineering> .",
)

# --- DEIX canonical ---
deix = Holon(
    iri="urn:holon:deix:canonical",
    label="DEIX Canonical",
    depth=0,
    boundary_ttl="""
        @prefix deix: <https://semantic.incose.org/DEIX_Ontology#> .
        <urn:shapes:deix:ArtifactShape> a sh:NodeShape ;
            sh:targetClass deix:Digital_Artifact ;
            sh:property [
                sh:path rdfs:label ;
                sh:minCount 1 ;
                sh:severity sh:Violation
            ] .
    """,
)

# --- Simulation tool ---
sim = Holon(
    iri="urn:holon:tool:sim",
    label="Simulation (AFSIM)",
    depth=1,
    interior_ttl="",
    boundary_ttl="""
        @prefix sim: <urn:sim:> .
        <urn:shapes:sim:InputShape> a sh:NodeShape ;
            sh:targetClass sim:InputBlock ;
            sh:property [
                sh:path rdfs:label ;
                sh:minCount 1 ;
                sh:severity sh:Violation
            ] .
    """,
    context_ttl="<urn:holon:tool:sim> cga:memberOf <urn:holon:domain:engineering> .",
)

# --- Portals ---

# Requirements → DEIX
portal_req_deix = TransformPortal(
    iri="urn:portal:doors-to-deix",
    source=reqs,
    target=deix,
    label="DOORS → DEIX",
    construct_query="""
        PREFIX req:  <urn:req:>
        PREFIX deix: <https://semantic.incose.org/DEIX_Ontology#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        CONSTRUCT {
            ?req a deix:Digital_Artifact .
            ?req rdfs:label ?label .
            ?req deix:reqText ?text .
            ?req deix:threshold ?thresh .
        }
        WHERE {
            ?req a req:Requirement .
            ?req rdfs:label ?label .
            OPTIONAL { ?req req:text ?text }
            OPTIONAL { ?req req:threshold ?thresh }
        }
    """,
)

# Design → DEIX
portal_design_deix = TransformPortal(
    iri="urn:portal:sysml-to-deix",
    source=design,
    target=deix,
    label="SysML → DEIX",
    construct_query="""
        PREFIX sysml: <urn:sysml:>
        PREFIX deix:  <https://semantic.incose.org/DEIX_Ontology#>
        PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>
        CONSTRUCT {
            ?block a deix:Digital_Artifact .
            ?block rdfs:label ?name .
            ?block deix:hasParameter ?param .
            ?param a deix:Digital_Information .
            ?param rdfs:label ?pname .
            ?param deix:informationValue ?pval .
            ?param deix:informationUnit ?punit .
        }
        WHERE {
            ?block a sysml:Block .
            ?block sysml:name ?name .
            ?param sysml:owner ?block .
            ?param sysml:name ?pname .
            ?param sysml:value ?pval .
            OPTIONAL { ?param sysml:unit ?punit }
        }
    """,
)

# DEIX → Simulation
portal_deix_sim = TransformPortal(
    iri="urn:portal:deix-to-sim",
    source=deix,
    target=sim,
    label="DEIX → Sim",
    construct_query="""
        PREFIX deix: <https://semantic.incose.org/DEIX_Ontology#>
        PREFIX sim:  <urn:sim:>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        CONSTRUCT {
            ?block a sim:InputBlock .
            ?block rdfs:label ?name .
            ?block sim:hasInput ?param .
            ?param a sim:InputParam .
            ?param rdfs:label ?pname .
            ?param sim:numericValue ?pval .
        }
        WHERE {
            ?block a deix:Digital_Artifact .
            ?block rdfs:label ?name .
            OPTIONAL {
                ?block deix:hasParameter ?param .
                ?param rdfs:label ?pname .
                ?param deix:informationValue ?pval .
            }
        }
    """,
)

print("  Holons: DOORS, SysML, DEIX (canonical), Simulation")
print("  Portals: DOORS→DEIX, SysML→DEIX, DEIX→Sim")

  Holons: DOORS, SysML, DEIX (canonical), Simulation
  Portals: DOORS→DEIX, SysML→DEIX, DEIX→Sim


## Execute the Pipeline With Full Provenance

Every portal traversal is recorded as a `prov:Activity`.
The provenance activities reference graph IRIs — creating
the cross-layer hyperedges that constitute the thread.

In [3]:
prov = ProvenanceTracker(
    agent_iri="urn:agent:deix-thread-manager",
    agent_label="DEIX Thread Manager",
)

# Leg 1: Requirements → DEIX
req_projection = portal_req_deix.traverse(reqs.interior)
for t in req_projection:
    deix.interior.add(t)
act1 = prov.record_traversal(
    portal_iri=portal_req_deix.iri,
    source=reqs, target=deix,
    notes="Requirements lifted to DEIX canonical form.",
)
print(f"  Leg 1: DOORS → DEIX  ({len(req_projection)} triples)  activity: {act1.split(':')[-1]}")

# Leg 2: Design → DEIX
design_projection = portal_design_deix.traverse(design.interior)
for t in design_projection:
    deix.interior.add(t)
act2 = prov.record_traversal(
    portal_iri=portal_design_deix.iri,
    source=design, target=deix,
    notes="Design model lifted to DEIX canonical form.",
)
print(f"  Leg 2: SysML → DEIX  ({len(design_projection)} triples)  activity: {act2.split(':')[-1]}")

# Leg 3: DEIX → Simulation
sim_input = portal_deix_sim.traverse(deix.interior)
for t in sim_input:
    sim.interior.add(t)
act3 = prov.record_traversal(
    portal_iri=portal_deix_sim.iri,
    source=deix, target=sim,
    notes="DEIX canonical data lowered to simulation input format.",
)
print(f"  Leg 3: DEIX → Sim    ({len(sim_input)} triples)  activity: {act3.split(':')[-1]}")

# Validate at the sim boundary
val_result = validate_membrane(sim)
prov.record_validation(
    holon=sim,
    conforms=val_result.conforms,
    health=val_result.health.value,
)
print(f"\n  Sim membrane: {val_result.health.value.upper()}")

  Leg 1: DOORS → DEIX  (4 triples)  activity: 6ae4deee1142
  Leg 2: SysML → DEIX  (7 triples)  activity: 2c2581727316
  Leg 3: DEIX → Sim    (8 triples)  activity: 3b20ef0368fe

  Sim membrane: INTACT


## The Digital Thread IS the Provenance Chain

Now we query the provenance across all context graphs.  The thread
is not a separate entity we constructed — it is the emergent structure
of the portal traversals we just executed.

In [5]:
# Merge all context graphs to query the thread
thread_graph = Graph()
for h in [reqs, design, deix, sim]:
    for t in h.context:
        thread_graph.add(t)

# Also add the DEIX ontology fragment needed for inference
thread_graph.parse(StringIO(PREFIXES + """
    deix:Digital_Thread_Role a rdfs:Class ;
        rdfs:subClassOf rdf:Statement ;
        rdfs:comment "Per DEIX: realized in Act of Measuring Consistency." .
"""), format="turtle")

# Query: the provenance chain IS the thread
thread_query = """
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX cga:  <urn:cga:>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?activity ?label ?time ?used ?generated ?portal
WHERE {
    ?activity a prov:Activity .
    ?activity rdfs:label ?label .
    ?activity prov:startedAtTime ?time .
    OPTIONAL { ?activity prov:used ?used }
    OPTIONAL { ?activity prov:generated ?generated }
    OPTIONAL { ?activity cga:viaPortal ?portal }
}
ORDER BY ?time
"""

print("  The Digital Thread — every traversal activity in sequence:\n")
print(f"  {'Activity':<15s} {'Label':<35s} {'Portal'}")
print(f"  {'-'*15} {'-'*35} {'-'*25}")

for row in thread_graph.query(thread_query):
    act_short = str(row[0]).split(":")[-1][:15]
    label = str(row[1])[:35]
    portal = str(row[5]).split(":")[-1] if row[5] else "—"
    print(f"  {act_short:<15s} {label:<35s} {portal}")

  The Digital Thread — every traversal activity in sequence:

  Activity        Label                               Portal
  --------------- ----------------------------------- -------------------------
  6ae4deee1142    Portal traversal: DOORS (Requiremen doors-to-deix
  2c2581727316    Portal traversal: SysML v2 (Design) sysml-to-deix
  3b20ef0368fe    Portal traversal: DEIX Canonical →  deix-to-sim
  f79a61a7afd3    Membrane validation: Simulation (AF —
  f79a61a7afd3    Membrane validation: Simulation (AF —


## Querying the Derivation Chain (The Hyperedges)

The `prov:wasDerivedFrom` triples connect graph IRIs across holons.
This is the actual thread structure — each derivation link says
"this graph's content came from that graph's content, via this portal."

In [6]:
derivation_query = """
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?derived ?source
WHERE {
    ?derived prov:wasDerivedFrom ?source .
}
ORDER BY ?derived
"""

print("  prov:wasDerivedFrom links between named graph IRIs:\n")
for row in thread_graph.query(derivation_query):
    derived = str(row[0]).replace("urn:holon:", "")
    source = str(row[1]).replace("urn:holon:", "")
    print(f"    {derived:35s} ← derived from ← {source}")

print("""
  These are the HYPEREDGES.  Each prov:wasDerivedFrom triple sits in
  a context graph (named graph C) and connects two interior graph IRIs
  (named graphs A and B).  The triple is IN graph C, but ABOUT graphs
  A and B.  That is the named-graph-as-participant pattern.

  The Digital Thread is this derivation chain.  It does not need to be
  separately described or maintained — it IS the provenance of the
  portal traversals that moved data through the DEIX canonical hub.
""")

  prov:wasDerivedFrom links between named graph IRIs:

    deix:canonical/interior             ← derived from ← tool:sysml/interior
    deix:canonical/interior             ← derived from ← tool:doors/interior
    tool:sim/interior                   ← derived from ← deix:canonical/interior

  These are the HYPEREDGES.  Each prov:wasDerivedFrom triple sits in
  a context graph (named graph C) and connects two interior graph IRIs
  (named graphs A and B).  The triple is IN graph C, but ABOUT graphs
  A and B.  That is the named-graph-as-participant pattern.

  The Digital Thread is this derivation chain.  It does not need to be
  separately described or maintained — it IS the provenance of the
  portal traversals that moved data through the DEIX canonical hub.



## Applying the Digital Thread Role (DEIX Alignment)

Per the DEIX Ontology:
- `deix:Digital_Thread_Role` is realized in `deix:Act_of_Measuring_Consistency`
- The Act of Measuring Consistency produces a `deix:Consistency_Measurement_ICE`
- The Thread Description ICE is composed of these measurements

In holonic terms:
- The `Digital_Thread_Role` is borne by the **collection of portals**
- Each portal traversal IS an `Act_of_Measuring_Consistency` (because
  the portal validates its output against the target boundary)
- Each validation result IS a `Consistency_Measurement_ICE`
- The thread IS the composition of these measurements

In [7]:
# Record the thread role in the DEIX canonical holon's context
deix.load_context(f"""
    @prefix deix: <https://semantic.incose.org/DEIX_Ontology#> .

    # The portal collection bears the Digital Thread Role
    <urn:thread:thermal-subsystem> a deix:Thread_Description_Information_Content_Entity ;
        rdfs:label "Thermal Subsystem Digital Thread" ;

        # Per DEIX: composed of traceability measurements
        # Each portal traversal + validation IS a measurement
        deix:composedOf <{act1}>, <{act2}>, <{act3}> ;

        # Per DEIX: participates in lifecycle
        deix:inLifecycle <urn:lifecycle:vehicle-v2> .

    # The Digital Thread Role is realized in the traversal activities
    <urn:role:thread:thermal> a deix:Digital_Thread_Role ;
        rdfs:label "Thermal Subsystem Thread Role" ;
        # Per DEIX: realized in Act of Measuring Consistency
        # Each portal traversal that validates = measuring consistency
        deix:realizedIn <{act1}>, <{act2}>, <{act3}> .

    # Each activity is both a prov:Activity AND a DEIX consistency act
    <{act1}> a deix:Act_of_Measuring_Consistency .
    <{act2}> a deix:Act_of_Measuring_Consistency .
    <{act3}> a deix:Act_of_Measuring_Consistency .
""")

print("  Digital Thread Role applied to the portal traversal chain:")
print(f"    Thread: urn:thread:thermal-subsystem")
print(f"    Role:   urn:role:thread:thermal")
print(f"    Composed of: {act1.split(':')[-1]}, {act2.split(':')[-1]}, {act3.split(':')[-1]}")
print()
print("  The thread is NOT a separate graph of trace links.")
print("  It IS the ROLE applied to the collection of portal traversals")
print("  and their provenance activities.  The derivation chain IS the thread.")

  Digital Thread Role applied to the portal traversal chain:
    Thread: urn:thread:thermal-subsystem
    Role:   urn:role:thread:thermal
    Composed of: 6ae4deee1142, 2c2581727316, 3b20ef0368fe

  The thread is NOT a separate graph of trace links.
  It IS the ROLE applied to the collection of portal traversals
  and their provenance activities.  The derivation chain IS the thread.


## The Full Architecture

```
DOORS Interior ──portal──→ DEIX Interior ──portal──→ Sim Interior
(req: vocab)    CONSTRUCT   (deix: vocab)  CONSTRUCT   (sim: vocab)
     │                           │                          │
     └───── context ─────────────┼──────── context ─────────┘
            prov:Activity        │         prov:Activity
            prov:used ──────→ graph IRIs ←────── prov:used
            prov:generated ←─ graph IRIs ──→ prov:generated
                                 │
                     ┌───────────┴───────────┐
                     │   DIGITAL THREAD ROLE  │
                     │   Realized in the      │
                     │   prov:Activities       │
                     │   (which ARE Acts of    │
                     │   Measuring Consistency)│
                     └────────────────────────┘
```

The context graphs (with their prov:Activity triples referencing
interior graph IRIs) ARE the hyperedges.  The Digital Thread Role
IS applied to those activities.  Nothing is separately maintained.
The thread emerges from the governed portal traversals.